In [1]:
import ee
import geemap
ee.Authenticate()
ee.Initialize()

In [2]:
Map = geemap.Map()

# Panama boundary
countries = ee.FeatureCollection("FAO/GAUL/2015/level0")
panama_fc = countries.filter(ee.Filter.eq("ADM0_NAME", "Panama"))
panama_geom = panama_fc.geometry()

Map.centerObject(panama_geom, 7)

In [3]:
districts = ee.FeatureCollection("projects/deforestation-panama/assets/Districts")

In [4]:
# Reclassified Land Cover Image
sinia_prism = ee.Image("projects/deforestation-panama/assets/landuse_2021").clip(panama_geom)

lu_index = [10,20,30,40, 50, 60, 70, 80]

VisParams = {
    "min": 10,
    "max": 80, 
    "palette": [
        '32a65e',  # [10] Mature forest
        '32a65e',  # [20] Secondary forest
        '1f8d49',  # [30] Primary forest
        '7dc975',  # [40] Crops
        '04381d',  # [50] Plantation
        '026975',  # [60] Other vegetation
        '000000',  # [70] Other landuse
        '7a6c00',  # [80] Other forest 
       ]
    }
        
    # A flat list of pixel values to replace
from_list = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33]
    # A corresponding list of new values
to_list = [10, 20, 80, 80, 80, 80, 20, 20, 60, 60, 60, 70, 70, 50, 40, 50, 40, 40, 40, 50, 40, 40, 40, 40, 50, 60, 27, 28, 29, 70, 70, 70, 70]

    # for each forest age, mask the lulc of the year immediately preceding abandonment
lulc_aggregated = ee.Image()
remapped_band = sinia_prism.remap(from_list, to_list)

# Remapped image with single band
lulc_aggregated = sinia_prism.remap(from_list, to_list).rename('landuse')

# Map pixel values to string labels for readable DataFrame outputs
class_labels = {
    '10': 'Mature forest',
    '20': 'Secondary forest',
    '30': 'Primary forest',
    '40': 'Crops',
    '50': 'Plantation',
    '60': 'Other vegetation',
    '70': 'Other landuse',
    '80': 'Other forest',
    '27': 'Water surface',
    '28': 'Populated area',
    '29': 'Infrastructure'
}

In [5]:
print(districts.first().propertyNames().getInfo())

['Area_Ha', 'Distrito', 'ID_Distrit', 'Provincia', 'system:index', 'GlobalID']


In [6]:
# --------------------------------------------------
# Calculate dominant land-use class per district
# --------------------------------------------------

# Function to calculate the dominant land-use class
def get_dominant_class(feature):

    # Calculate the frequency of each land-use class
    histogram = lulc_aggregated.reduceRegion(
        reducer=ee.Reducer.frequencyHistogram(),
        geometry=feature.geometry(),
        scale=30,          # Adjust to the resolution of your image
        maxPixels=1e13,
        tileScale=4
    )

    # Extract the histogram dictionary
    hist = ee.Dictionary(histogram.get('landuse'))

    # Get the class with the highest pixel frequency
    dominant_class = ee.Number.parse(
        ee.List(hist.keys()).sort(
            ee.List(hist.values())
        ).get(-1)
    )

    # Convert numerical class to a readable label
    dominant_label = ee.Dictionary(class_labels).get(
        dominant_class.format('%d')
    )

    # Return the district name and dominant land-use class
    return feature.set({
        'dominant_class': dominant_class,
        'dominant_landuse': dominant_label
    })


# Apply function to all districts
districts_dominant = districts.map(get_dominant_class)

In [7]:
# Keep only district name and dominant land-use class
results = districts_dominant.select([
    'Distrito',
    'dominant_landuse'
])

In [9]:
import pandas as pd

# Retrieve district features from Google Earth Engine
features_info = districts_dominant.getInfo()['features']

# Extract district names and dominant land-use classes
rows = []

for f in features_info:
    props = f['properties']

    row = {
        'DISTRICT': props.get('Distrito'),
        'DOMINANT_LANDUSE': props.get('dominant_landuse')
    }

    rows.append(row)

# Create Pandas DataFrame
df = pd.DataFrame(rows)

# Sort alphabetically by district
df = df.sort_values(by='DISTRICT').reset_index(drop=True)

# Display dataframe
print(df)

         DISTRICT  DOMINANT_LANDUSE
0       Aguadulce  Other vegetation
1          Alanje             Crops
2       Almirante     Mature forest
3           Antón  Other vegetation
4        Arraiján     Mature forest
..            ...               ...
77         Taboga  Secondary forest
78  Tierras Altas  Other vegetation
79           Tolé  Other vegetation
80         Tonosí  Other vegetation
81          Ñürün  Other vegetation

[82 rows x 2 columns]


In [ ]:
task = ee.batch.Export.table.toDrive(
    collection=results,
    description='Panama_Dominant_Landuse_by_District',
    folder='GEE_Exports',
    fileNamePrefix='Panama_Dominant_Landuse_by_District',
    fileFormat='CSV',
    selectors=['Distrito', 'dominant_landuse']
)

task.start()

print("Export started!")